# Load data

In [ ]:
from collections.abc import Sequence
from pathlib import Path

import nibabel as nib
import numpy as np


def load_subject_maps(
    base_paths: Sequence[str | Path],
    relative_path: str | Path,
    extension: str = ".nii.gz",
    *,
    dtype: np.dtype = np.float32,
) -> np.ndarray:
    """
    Load the same image map for multiple subjects and stack subjects
    along the last array dimension.

    Parameters
    ----------
    base_paths:
        Subject-specific base directories. Their order determines the
        order along the final subject dimension.

    relative_path:
        Path to the map relative to each base directory, without the
        file extension.

        Example:
            "Extra/FWHM_map"

    extension:
        File extension, for example ".nii.gz" or ".nii".

    dtype:
        NumPy dtype of the returned array.

    Returns
    -------
    np.ndarray
        Array with shape:

            (*spatial_shape, n_subjects)

        For example:

            (64, 64, 35, 9)

    Raises
    ------
    FileNotFoundError:
        If one of the requested files does not exist.

    ValueError:
        If no base paths were supplied or the image shapes differ.
    """
    base_paths = [
        Path(base_path)
        for base_path in base_paths
    ]

    if not base_paths:
        raise ValueError(
            "base_paths must contain at least one subject directory."
        )

    if not extension.startswith("."):
        extension = f".{extension}"

    relative_file = Path(
        f"{relative_path}{extension}"
    )

    subject_arrays: list[np.ndarray] = []
    expected_shape: tuple[int, ...] | None = None

    for base_path in base_paths:
        file_path = base_path / relative_file

        if not file_path.is_file():
            raise FileNotFoundError(
                "Subject map not found:\n"
                f"  {file_path}"
            )

        image = nib.load(
            str(file_path)
        )

        array = np.asarray(
            image.get_fdata(),
            dtype=dtype,
        )

        if expected_shape is None:
            expected_shape = array.shape

        elif array.shape != expected_shape:
            raise ValueError(
                "All subject maps must have the same shape:\n"
                f"  expected: {expected_shape}\n"
                f"  found:    {array.shape}\n"
                f"  file:     {file_path}"
            )

        subject_arrays.append(
            array
        )

    return np.stack(
        subject_arrays,
        axis=-1,
    )

In [ ]:
SUBJECT_DIRS = [
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol03_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol04_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol05_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol07_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol01_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol02_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol03_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol04_Dat_NoL2_GradDel/maps",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol05_Dat_NoL2_GradDel/maps"
]

In [ ]:
fwhm_maps = load_subject_maps(
    base_paths=SUBJECT_DIRS,
    relative_path="Extra/SNR_map",
    extension=".nii.gz",
)

mask = load_subject_maps(
    base_paths=SUBJECT_DIRS,
    relative_path="mask",
    extension=".nii.gz",
)

# Create masked 1D and subject wise pools for statistical evaluation

In [ ]:
from __future__ import annotations

import numpy as np


def extract_valid_voxels(
    maps: np.ndarray,
    quality_mask: np.ndarray,
) -> tuple[dict[str, np.ndarray], np.ndarray]:
    """
    Extract valid voxel values separately for each subject and pool
    them across all subjects.

    Parameters
    ----------
    maps:
        Parameter maps with shape:

            (x, y, z, n_subjects)

    quality_mask:
        Binary quality mask with the same shape as ``maps``.
        Voxels with value 1 are included; voxels with value 0 are
        excluded.

    Returns
    -------
    subject_values:
        Dictionary containing one one-dimensional NumPy array per
        subject. The keys are strings:

            {
                "0": values_subject_0,
                "1": values_subject_1,
                ...
            }

    pooled_values:
        One-dimensional NumPy array containing all valid voxel values
        from all subjects, concatenated in subject order.
    """
    maps = np.asarray(maps)
    quality_mask = np.asarray(quality_mask)

    if maps.ndim != 4:
        raise ValueError(
            "maps must have shape (x, y, z, n_subjects), "
            f"but found shape {maps.shape}."
        )

    if quality_mask.shape != maps.shape:
        raise ValueError(
            "maps and quality_mask must have the same shape:\n"
            f"  maps:         {maps.shape}\n"
            f"  quality_mask: {quality_mask.shape}"
        )

    mask_is_finite = np.isfinite(quality_mask)

    unique_mask_values = np.unique(
        quality_mask[mask_is_finite]
    )

    if not np.all(
        np.isin(
            unique_mask_values,
            [0, 1],
        )
    ):
        raise ValueError(
            "quality_mask may contain only 0 and 1, "
            f"but found values {unique_mask_values}."
        )

    subject_values: dict[str, np.ndarray] = {}

    n_subjects = maps.shape[-1]

    for subject_index in range(n_subjects):
        subject_map = maps[..., subject_index]
        subject_mask = quality_mask[..., subject_index]

        valid = (
            (subject_mask == 1)
            & np.isfinite(subject_map)
        )

        values = subject_map[valid].reshape(-1)

        subject_values[str(subject_index)] = values

    nonempty_arrays = [
        values
        for values in subject_values.values()
        if values.size > 0
    ]

    if nonempty_arrays:
        pooled_values = np.concatenate(
            nonempty_arrays,
            axis=0,
        )
    else:
        pooled_values = np.empty(
            0,
            dtype=maps.dtype,
        )

    return subject_values, pooled_values

In [ ]:
subject_fwhm, pooled_fwhm = extract_valid_voxels(
    maps=fwhm_maps,
    quality_mask=mask,
)

# Compute median and IQR for the pool

In [ ]:
import numpy as np


def calculate_pooled_median_iqr(
    pooled_values: np.ndarray,
) -> tuple[float, float]:
    """
    Calculate the median and interquartile range of pooled voxel values.

    Parameters
    ----------
    pooled_values:
        One-dimensional array containing pooled voxel values from all
        subjects.

    Returns
    -------
    median:
        Median of all pooled values.

    iqr:
        Interquartile range:

            IQR = Q3 - Q1
    """
    pooled_values = np.asarray(
        pooled_values,
        dtype=np.float64,
    )

    if pooled_values.ndim != 1:
        raise ValueError(
            "pooled_values must be one-dimensional, "
            f"but found shape {pooled_values.shape}."
        )

    pooled_values = pooled_values[
        np.isfinite(pooled_values)
    ]

    if pooled_values.size == 0:
        raise ValueError(
            "pooled_values contains no finite values."
        )

    q1, median, q3 = np.percentile(
        pooled_values,
        [25, 50, 75],
    )

    iqr = q3 - q1

    return float(median), float(iqr)

In [ ]:
median_fwhm, iqr_fwhm = calculate_pooled_median_iqr(
    pooled_fwhm
)

print(f"Median: {median_fwhm:.4f}")
print(f"IQR:    {iqr_fwhm:.4f}")

# Histogramm + Normalverteilungsplot

In [ ]:
from __future__ import annotations

from math import erf, pi, sqrt
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np


DistributionModel = Literal[
    "normal",
    "truncated_normal",
]


def plot_pooled_histogram_with_model(
    pooled_values: np.ndarray,
    *,
    mean: float,
    std: float,
    distribution: DistributionModel = "normal",
    xlabel: str,
    ylabel: str = "Probability density",
    title: str | None = None,
    bins: int | str = 50,
    x_limits: tuple[float, float] | None = None,
    save_path: str | Path | None = None,
    dpi: int = 300,
    n_model_points: int = 1000,
) -> tuple[plt.Figure, plt.Axes]:
    """
    Plot a density-normalized histogram of pooled voxel values and
    overlay a probability-density model.

    Parameters
    ----------
    pooled_values:
        One-dimensional array containing pooled values from all
        subjects.

    mean:
        Mean of the overlaid normal distribution.

    std:
        Standard deviation of the overlaid normal distribution.
        Must be strictly positive.

    distribution:
        Model to overlay:

        "normal"
            Ordinary normal distribution.

        "truncated_normal"
            Normal distribution truncated at zero. Negative samples
            are assumed to be rejected and newly sampled.

    xlabel:
        Label of the x-axis, including the unit.

    ylabel:
        Label of the y-axis.

    title:
        Optional title.

    bins:
        Number of histogram bins or a NumPy binning strategy such as
        "auto", "fd", or "sturges".

    x_limits:
        Optional x-axis limits as (minimum, maximum).

    save_path:
        Optional path under which the figure is saved.

        Examples:
            "figures/fwhm_histogram.png"
            Path("/data/results/fwhm_histogram.pdf")

        The parent directory is created automatically.

    dpi:
        Resolution used when saving raster formats such as PNG.

    n_model_points:
        Number of points used to draw the model density.

    Returns
    -------
    fig:
        Matplotlib figure.

    ax:
        Matplotlib axes.
    """
    values = np.asarray(
        pooled_values,
        dtype=np.float64,
    )

    if values.ndim != 1:
        raise ValueError(
            "pooled_values must be one-dimensional, "
            f"but found shape {values.shape}."
        )

    values = values[
        np.isfinite(values)
    ]

    if values.size == 0:
        raise ValueError(
            "pooled_values contains no finite values."
        )

    mean = float(mean)
    std = float(std)

    if not np.isfinite(mean):
        raise ValueError(
            "mean must be finite."
        )

    if not np.isfinite(std) or std <= 0:
        raise ValueError(
            "std must be finite and greater than zero."
        )

    if distribution not in {
        "normal",
        "truncated_normal",
    }:
        raise ValueError(
            "distribution must be either "
            "'normal' or 'truncated_normal'."
        )

    if n_model_points < 2:
        raise ValueError(
            "n_model_points must be at least 2."
        )

    if x_limits is None:
        data_min = float(np.min(values))
        data_max = float(np.max(values))

        model_min = mean - 4.0 * std
        model_max = mean + 4.0 * std

        if distribution == "truncated_normal":
            model_min = 0.0

        x_min = min(
            data_min,
            model_min,
        )

        x_max = max(
            data_max,
            model_max,
        )

        if x_min == x_max:
            margin = max(
                abs(x_min) * 0.05,
                1.0,
            )

            x_min -= margin
            x_max += margin

    else:
        x_min, x_max = map(
            float,
            x_limits,
        )

        if (
            not np.isfinite(x_min)
            or not np.isfinite(x_max)
            or x_min >= x_max
        ):
            raise ValueError(
                "x_limits must contain two finite values "
                "with minimum < maximum."
            )

    x = np.linspace(
        x_min,
        x_max,
        n_model_points,
    )

    z = (
        x - mean
    ) / std

    normal_pdf = (
        np.exp(
            -0.5 * z**2
        )
        / (
            std * sqrt(2.0 * pi)
        )
    )

    if distribution == "normal":
        model_pdf = normal_pdf

        model_label = (
            f"Normal model: μ={mean:.4g}, σ={std:.4g}"
        )

    else:
        lower_z = (
            0.0 - mean
        ) / std

        probability_above_zero = (
            1.0
            - 0.5
            * (
                1.0
                + erf(
                    lower_z / sqrt(2.0)
                )
            )
        )

        if probability_above_zero <= 0:
            raise ValueError(
                "The truncated normal distribution has "
                "numerically zero probability above zero."
            )

        model_pdf = np.where(
            x >= 0.0,
            normal_pdf / probability_above_zero,
            0.0,
        )

        model_label = (
            "Zero-truncated normal model: "
            f"μ={mean:.4g}, σ={std:.4g}"
        )

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.hist(
        values,
        bins=bins,

        # Dichtenormiertes Histogramm:
        # Die gesamte Fläche unter den Balken ist 1.
        density=True,

        label="Pooled voxel values",
    )

    ax.plot(
        x,
        model_pdf,
        linewidth=2,
        label=model_label,
    )

    ax.set_xlim(
        x_min,
        x_max,
    )

    ax.set_xlabel(
        xlabel
    )

    ax.set_ylabel(
        ylabel
    )

    if title is not None:
        ax.set_title(
            title
        )

    ax.legend()

    fig.tight_layout()

    if save_path is not None:
        save_path = Path(
            save_path
        )

        save_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight",
        )

    return fig, ax

In [ ]:
fig, ax = plot_pooled_histogram_with_model(
    pooled_values=pooled_fwhm,
    mean=median_fwhm,
    std=2*iqr_fwhm,
    distribution="truncated_normal",
    xlabel="FWHM [ppm]",
    title="Pooled FWHM distribution",
    bins=60,
    save_path="SavedGraphics/fwhm_histogram.png",
)

plt.show()